In [12]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/house-prices-advanced-regression-techniques


In [15]:
import os
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

train_df = pd.read_csv(os.path.join(path, 'train.csv'))
test_df = pd.read_csv(os.path.join(path, 'test.csv'))

print(f"Training data: {train_df.shape}")
print(f"Testing : {test_df.shape}")

Training data: (1460, 81)
Testing : (1459, 80)


In [8]:

y = np.log1p(train_df['SalePrice'])


X = train_df.drop(['Id', 'SalePrice'], axis=1)
X_test = test_df.drop(['Id'], axis=1)

In [9]:

all_features = pd.concat([X, X_test], axis=0).reset_index(drop=True)


num_cols = all_features.select_dtypes(include=[np.number]).columns
cat_cols = all_features.select_dtypes(include=['object']).columns


all_features[num_cols] = all_features[num_cols].fillna(all_features[num_cols].median())
all_features[cat_cols] = all_features[cat_cols].fillna('None')


all_features = pd.get_dummies(all_features, drop_first=True)


X_processed = all_features.iloc[:len(train_df)].copy()
X_test_processed = all_features.iloc[len(train_df):].copy()

In [16]:

X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)
model = XGBRegressor(n_estimators=500,learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [18]:

val_preds_log = model.predict(X_val)

rmse_log = np.sqrt(mean_squared_error(y_val, val_preds_log))


y_val_dollars = np.expm1(y_val)
val_preds_dollars = np.expm1(val_preds_log)
rmse_dollars = np.sqrt(mean_squared_error(y_val_dollars, val_preds_dollars))

print(f"Validation Log-RMSE (For Kaggle): {rmse_log:.4f}")
print(f"Validation RMSE Doolar: ${rmse_dollars:,.2f}")

Validation Log-RMSE (For Kaggle): 0.1435
Validation RMSE Doolar: $26,602.75


In [20]:

preds_log = model.predict(X_test_processed)

preds = np.expm1(preds_log)


submission = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': preds})
submission.to_csv('xgbmodel_houseproblem.csv', index=False)

print("Submission.csv Done!")

Submission.csv Done!


In [10]:
submission = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)